# SRA 시냅스 삭제 데모
## — LLM에서 특정 지식을 물리적으로 추출합니다!

이 노트북은 *[\[AI Romance\] LLM에서 특정 지식을 물리적으로 추출! 기사의 대화형 데모입니다. 핫스왑 가능 LLM의 '시냅스 삭제' 및 '퍼지'](https://qiita.com/JunSuzukiJapan/items/)*.

SRA(Synaptic Routing Architecture)에서는 불필요한 지식과 Synapse를 두 가지 방법으로 제거할 수 있습니다.

| 방법 | API | 목적 |
|------|------|------|
| **물리적 제거** | `pop_synapses(N)` | Hot-Swap을 통해 추가된 시냅스를 꼬리에서 제거하여 모델 크기 복원 |
| **제로클리어(퍼지)** | `clear_synapses([idx])` | 중간 인덱스에서 시냅스를 비활성화하고 이를 여유 슬롯으로 변환 |

또한 제로 클리어링 시 발생하는 **코사인 유사성 트랩**과 그 해결 방법을 실제로 확인해 보겠습니다.

마지막으로 실제로 다중 작업 훈련 모델에 `clear_synapses`를 적용하고 **대상 작업의 기능만 파괴되고 다른 모든 작업은 완전히 그대로 유지됩니다(Zero Forgetting)**를 보여줍니다.

---
## 파트 1: 시냅스 제거(`pop_synapses`)

나중에 Hot-Swap을 통해 추가된 시냅스를 꼬리부터 물리적으로 잘라냈습니다.
OS에서 플러그인을 제거하는 것과 마찬가지로 AI 두뇌의 일부를 물리적으로 제거할 수 있습니다.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# Build a small model to walk through the add -> remove flow
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== Approach 1: Synapse Removal (Physical Cut) ===')
print(f'[Initial state] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Add 2 Synapses via Hot-Swap
model.add_synapses(2, freeze_base=True)
print(f'\n[After adding] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Physically remove the added Synapses
model.pop_synapses(2)
print(f'\n[After removal] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')
print('\nMemory usage has been fully restored!')


---
## 2부: 제로 클리어링(퍼지) 및 "코사인 유사성 트랩"

중간 인덱스에서 Synapse를 삭제하려는 경우 이를 물리적으로 제거하면 ID가 이동하게 됩니다.
따라서 대신 "제로 클리어"인 `0.0`로 가중치를 덮어씁니다.

그러나 여기에는 **무서운 함정**이 있습니다. 영 벡터의 코사인 유사성은 `0.0`가 되며,
그리고 정상적인 시냅스의 점수가 음수이면 **빈 시냅스의 점수가 더 높아져 데이터를 흡수하기 시작합니다**. 이는 블랙홀 현상입니다.

SRA의 라우터에는 이러한 함정을 방지하기 위해 **`-inf` 마스크**가 내장되어 있습니다. 실제로 확인해 보겠습니다.

In [ ]:
print('=== Approach 2: Verifying zero-clear and the -inf mask ===')

# Create a new model
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# Inspect the weights of Synapse #2 before zero-clearing
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[Before zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# Execute zero-clear
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[After zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\nThe weights of Synapse #2 have been zeroed out completely!')


In [ ]:
# Actually verify the -inf mask in action
print('=== Verifying the -inf mask ===')

# Run the router with a dummy input
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# Inspect the router logits (final layer)
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'Router scores (first token):')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = 'BLOCKED (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' <- zero-cleared' if i == target_idx else ''
    print(f'  Synapse {i}: {label}{marker}')

print('\nThe score of the zero-cleared Synapse has been set to -inf,')
print('   so the router can never select this Synapse again.')
print('   This completely prevents the "black-hole phenomenon".')


---
## 3부: 기능 증명 — 훈련된 모델에 대한 `clear_synapses`

이제 메인 이벤트에 들어갑니다. 파트 1과 2에서는 "API 동작"을 확인했습니다.
하지만 정말 중요한 것은 기능적 증명입니다. **"제로 클리어 후 삭제된 지식만 손실되고 다른 모든 지식은 그대로 유지됩니까?"**

노트북 05(Lesion 실험)와 동일한 접근 방식을 사용합니다.
1. `copy` 및 `reverse`의 멀티태스킹 훈련
2. `reverse`에서 자주 사용하는 시냅스를 식별하고 `clear_synapses`로 제거합니다.
3. `copy`가 계속해서 100%(Zero Forgetting) 점수를 기록하는 동안 `reverse`가 해결 불가능 상태가 되는지 확인합니다.

In [ ]:
# Exactly the same setup as notebook 05
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# Multitask training (same as notebook 05)
print('Training started... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('Training finished!')


### 3-1. 삭제 전 성능 테스트 및 대상 식별
우리는 각 작업이 올바르게 해결될 수 있는지 확인하고 `reverse`에서 가장 많이 사용되는 시냅스를 식별합니다.

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # Aggregate which Synapses were used (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] Accuracy: {acc*100:.1f}%')
    return usage

print('=== Before Deletion ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Identify Synapses heavily used by Reverse but rarely used by Copy
# (Same logic as notebook 05)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# If they cannot be perfectly separated, pick the single Synapse most used by Reverse
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> Deletion target: Synapses {target_synapses} (heavily used by REVERSE)')


### 3-2. `clear_synapses`를 통한 시냅스 삭제
노트북 05에서는 `block.w1.data[idx].zero_()`를 수동으로 실행했지만 여기서는 안전한 삭제를 수행하기 위해 라우터의 `-inf` 마스크도 적용하는 공식 **`clear_synapses` API**를 사용합니다.

In [ ]:
print(f'Deleting Synapses {target_synapses} via clear_synapses...')

# Record norms before deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [Before deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')

# Use the clear_synapses API (the router's -inf mask is also applied automatically)
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\nDeletion complete!')

# Check norms after deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [After deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')


### 3-3. 삭제 후 성능 테스트(Zero Forgetting 검증)

일부 시냅스를 제거한 상태에서 다시 테스트합니다.

**예상 결과:**
- **복사**: 정확성 유지(다른 시냅스를 사용하므로 완전히 손상되지 않음)
- **역방향**: 정확도가 떨어집니다(특수 시냅스가 사라졌습니다).

In [ ]:
print('=== After Deletion ===')
test_task('copy')
test_task('reverse')

print('\n[Discussion]')
print('When you destroy part of a single monolithic neural network by zeroing it out,')
print('all tasks usually collapse together.')
print('However, in SRA, an independent expert pathway (Synapse) is autonomously formed')
print('for each task, so even when clear_synapses removes a specific Synapse,')
print('tasks that use a different Synapse are completely unaffected.')
print('\nThis is the true strength of SRA as a modular AI.')
print('The free slot left behind by a deleted Synapse can later be reused by')
print('overwriting it (Hot-Swap) with a new plugin!')


---
## 요약

| 데모 | 시연된 내용 |
|------|---------|
| 1부 | `pop_synapses`를 통한 물리적 제거 및 메모리 복원 |
| 2부 | `clear_synapses`를 통한 제로 클리어링 및 `-inf` 마스크 검증 |
| 3부 | 훈련된 모델의 `clear_synapses` -> 대상 작업만 파괴되고 다른 작업은 그대로 유지 |

이를 통해 우리는 모듈형 AI의 전체 수명 주기인 **"훈련 -> 통합(핫 스왑) -> 삭제(퍼지) -> 슬롯 재사용(덮어쓰기)"**을 달성했습니다.